In [1]:
import math
from collections import Counter

# ============================================================
# STEP 1: DATA  (small corpus split into train and test)
# ============================================================
train = [
    "i like nlp",
    "i like python",
    "we like nlp",
    "we study python",
]
test = [
    "we like python",
    "i study nlp",
]

print("=" * 60)
print("STEP 1: DATA")
print("=" * 60)
print("Training data:", train)
print("Testing data :", test)

# ============================================================
# STEP 2: TOKENIZE  (add <s> start and </s> end markers)
# ============================================================
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split() + ["</s>"]

print("\n" + "=" * 60)
print("STEP 2: TOKENIZE")
print("=" * 60)
for s in train:
    print(f"  {s!r:22} -> {tokenize(s)}")

# ============================================================
# STEP 3: N-GRAMS  (unigram, bigram, trigram)
# ============================================================
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

print("\n" + "=" * 60)
print("STEP 3: N-GRAMS for the sentence 'i like nlp'")
print("=" * 60)
tokens = tokenize("i like nlp")
print("Unigrams:", get_ngrams(tokens, 1))
print("Bigrams :", get_ngrams(tokens, 2))
print("Trigrams:", get_ngrams(tokens, 3))

# ============================================================
# STEP 4: COUNT n-grams in the TRAINING data only
# ============================================================
uni, bi, tri = Counter(), Counter(), Counter()
for s in train:
    t = tokenize(s)
    uni.update(get_ngrams(t, 1))
    bi.update(get_ngrams(t, 2))
    tri.update(get_ngrams(t, 3))

# Words the model can PREDICT = every word + </s>  (we never predict <s>)
predictable = [w for (w,) in uni if w != "<s>"]
V = len(predictable)
N = sum(c for (w,), c in uni.items() if w != "<s>")

print("\n" + "=" * 60)
print("STEP 4: COUNTS (from training data)")
print("=" * 60)
print("Unigram counts:", {w: c for (w,), c in uni.items()})
print("Bigram counts :")
for pair, c in sorted(bi.items(), key=lambda x: -x[1]):
    print(f"   {pair[0]:>5} {pair[1]:<6} : {c}")
print("Trigram counts:", {" ".join(k): c for k, c in tri.items()})
print(f"Vocabulary size V = {V}   |   total predicted tokens N = {N}")

# ============================================================
# STEP 5: PROBABILITY ESTIMATION (MLE = count and divide)
# ============================================================
def p_unigram(w):
    return uni[(w,)] / N

def p_bigram(prev, w):
    return bi[(prev, w)] / uni[(prev,)] if uni[(prev,)] else 0.0

def p_trigram(w1, w2, w3):
    return tri[(w1, w2, w3)] / bi[(w1, w2)] if bi[(w1, w2)] else 0.0

print("\n" + "=" * 60)
print("STEP 5: MLE PROBABILITIES")
print("=" * 60)
print(f"Unigram  P(nlp)            = {uni[('nlp',)]}/{N} = {p_unigram('nlp'):.4f}")
print(f"Bigram   P(like | i)       = {bi[('i','like')]}/{uni[('i',)]} = {p_bigram('i','like'):.4f}")
print(f"Bigram   P(nlp | like)     = {bi[('like','nlp')]}/{uni[('like',)]} = {p_bigram('like','nlp'):.4f}")
print(f"Trigram  P(nlp | i, like)  = {tri[('i','like','nlp')]}/{bi[('i','like')]} = {p_trigram('i','like','nlp'):.4f}")

def sentence_prob(sentence, prob_fn):
    p = 1.0
    for prev, w in get_ngrams(tokenize(sentence), 2):
        p *= prob_fn(prev, w)
    return p

print("\nSentence probability with the bigram model (MLE):")
for s in test:
    print(f"   P({s!r}) = {sentence_prob(s, p_bigram)}")
print("   -> 'i study nlp' = 0 because 'i study' and 'study nlp' were never seen. ZERO PROBLEM!")

# ============================================================
# STEP 6: SMOOTHING (Add-One / Laplace)
# ============================================================
def p_bigram_smooth(prev, w):
    return (bi[(prev, w)] + 1) / (uni[(prev,)] + V)

def p_unigram_smooth(w):
    return (uni[(w,)] + 1) / (N + V)

print("\n" + "=" * 60)
print("STEP 6: SMOOTHING (Add-One)")
print("=" * 60)
print("Formula: P(w | prev) = (Count(prev,w) + 1) / (Count(prev) + V)")
print(f"P(like | i)    MLE = {p_bigram('i','like'):.4f}  ->  Smoothed = "
      f"({bi[('i','like')]}+1)/({uni[('i',)]}+{V}) = {p_bigram_smooth('i','like'):.4f}")
print(f"P(study | i)   MLE = {p_bigram('i','study'):.4f}  ->  Smoothed = "
      f"({bi[('i','study')]}+1)/({uni[('i',)]}+{V}) = {p_bigram_smooth('i','study'):.4f}")
print(f"P(nlp | study) MLE = {p_bigram('study','nlp'):.4f}  ->  Smoothed = "
      f"({bi[('study','nlp')]}+1)/({uni[('study',)]}+{V}) = {p_bigram_smooth('study','nlp'):.4f}")

print("\nSentence probability after smoothing:")
for s in test:
    print(f"   P({s!r}) = {sentence_prob(s, p_bigram_smooth):.6f}")

# ============================================================
# STEP 7: PERPLEXITY on the TEST data (lower = better)
# ============================================================
def perplexity(sentences, prob_fn, use_bigram=True):
    log_sum, count = 0.0, 0
    for s in sentences:
        t = tokenize(s)
        pairs = get_ngrams(t, 2)
        for prev, w in pairs:
            p = prob_fn(prev, w) if use_bigram else prob_fn(w)
            if p == 0:
                return float("inf")
            log_sum += math.log(p)
            count += 1
    return math.exp(-log_sum / count)

print("\n" + "=" * 60)
print("STEP 7: PERPLEXITY on test data")
print("=" * 60)
print("Formula: PP = exp( -(1/N) * sum( log P(word) ) )")
print(f"Bigram  + MLE      : {perplexity(test, p_bigram)}")
print(f"Bigram  + Smoothing: {perplexity(test, p_bigram_smooth):.3f}")
print(f"Unigram + Smoothing: {perplexity(test, p_unigram_smooth, use_bigram=False):.3f}")

print("\nWord-by-word detail (bigram + smoothing) for 'we like python':")
log_sum, k = 0.0, 0
for prev, w in get_ngrams(tokenize("we like python"), 2):
    p = p_bigram_smooth(prev, w)
    log_sum += math.log(p); k += 1
    print(f"   P({w:>6} | {prev:>6}) = {p:.4f}")
print(f"   Perplexity of this sentence = exp({-log_sum:.3f}/{k}) = {math.exp(-log_sum / k):.3f}")

STEP 1: DATA
Training data: ['i like nlp', 'i like python', 'we like nlp', 'we study python']
Testing data : ['we like python', 'i study nlp']

STEP 2: TOKENIZE
  'i like nlp'           -> ['<s>', 'i', 'like', 'nlp', '</s>']
  'i like python'        -> ['<s>', 'i', 'like', 'python', '</s>']
  'we like nlp'          -> ['<s>', 'we', 'like', 'nlp', '</s>']
  'we study python'      -> ['<s>', 'we', 'study', 'python', '</s>']

STEP 3: N-GRAMS for the sentence 'i like nlp'
Unigrams: [('<s>',), ('i',), ('like',), ('nlp',), ('</s>',)]
Bigrams : [('<s>', 'i'), ('i', 'like'), ('like', 'nlp'), ('nlp', '</s>')]
Trigrams: [('<s>', 'i', 'like'), ('i', 'like', 'nlp'), ('like', 'nlp', '</s>')]

STEP 4: COUNTS (from training data)
Unigram counts: {'<s>': 4, 'i': 2, 'like': 3, 'nlp': 2, '</s>': 4, 'python': 2, 'we': 2, 'study': 1}
Bigram counts :
     <s> i      : 2
       i like   : 2
    like nlp    : 2
     nlp </s>   : 2
   python </s>   : 2
     <s> we     : 2
    like python : 1
      we like   :